# 00 · Project Setup
## Kenyan Sign Language (KSL) Recognition with Deep Learning

| | |
|---|---|
| **Course** | DSA 4050-XA: Deep Learning, USIU-Africa (Fall 2026) |
| **Lecturer** | Dr. Alex Wambua |
| **Student** | Selmah Tzindori (669602) |
| **Notebook** | `00_project_setup.ipynb` |

### Objective
Prepare the working environment for the whole project. Run this notebook **once** at the start (and again only if folders or the helper file need to be recreated).

### What this notebook does
1. Connects Google Drive.
2. Creates the project folder structure on Drive.
3. Writes the shared helper module `ksl_utils.py` (used by every other notebook).
4. Checks that a GPU is available.
5. Records the software environment (Python, TensorFlow, GPU) for the report's *Methodology* section.

### Outputs
- Folder tree under `MyDrive/DSA4050/ksl-recognition/`
- `src/ksl_utils.py`
- `outputs/results.json` with the environment details

### Where things are stored

Large files cannot go on GitHub (100 MB file limit, and datasets should not be in a code repo). So the project is split in two:

| Location | What goes there | Why |
|---|---|---|
| **Google Drive** `MyDrive/DSA4050/ksl-recognition/` | Raw and processed images, trained models, all outputs | Large files, persists between Colab sessions |
| **GitHub** `projects/ksl-recognition/` (managed in VS Code) | Notebooks, `ksl_utils.py`, dashboard code, report, slides, and copies of the final figures/tables | Version control and submission; small files only |

**Drive folder structure** (created below):
```
ksl-recognition/
├── data/
│   ├── raw/            original downloaded dataset, never edited
│   └── processed/      cleaned / resized images and train-val-test splits
├── models/
│   └── checkpoints/    best weights saved during training
├── outputs/
│   ├── figures/        PNG charts (300 dpi) for the report and slides
│   ├── tables/         CSV tables for the report
│   ├── logs/           training histories + index of every saved output
│   ├── results.json    key numbers quoted in the report
│   └── experiments.csv one row per trained model (model comparison table)
└── src/
    └── ksl_utils.py    shared helper functions
```

## Step 1 · Connect Google Drive
A pop-up will ask you to sign in and allow access. After this, Drive appears at `/content/drive/MyDrive/`.

In [ ]:
# Mount Google Drive so Colab can read and write files there
from google.colab import drive
drive.mount("/content/drive")

## Step 2 · Write the shared helper module
`ksl_utils.py` holds the functions every notebook uses, so that outputs are saved consistently:

| Function | What it does |
|---|---|
| `make_folders()` | Creates all project folders |
| `set_seed()` | Fixes random seeds so results can be reproduced |
| `save_figure(fig, name, notebook, caption)` | Saves a 300-dpi PNG to `outputs/figures/` and records its caption |
| `save_table(df, name, notebook, caption)` | Saves a CSV to `outputs/tables/` |
| `log_result(key, value, notebook)` | Stores one key number in `results.json` (e.g. total images) |
| `log_experiment(model, type, metrics, notebook, params)` | Adds a row to `experiments.csv` for each trained model |
| `model_path(name)` | Path on Drive for saving a model |
| `save_history()` / `plot_history()` | Saves training curves (accuracy/loss per epoch) |
| `restore_raw_data()` | Unzips the raw dataset backup onto Colab's local disk (once per session) |
| `restore_processed_data()` | Unzips the 224×224 images made in notebook 03 |
| `load_split_catalogue()` / `load_class_names()` | Final catalogue with train/val/test split; sign names in label order |
| `project_summary()` | Prints how many figures, tables, models and results exist |

The cell below first creates the `src/` folder, then writes the file into it.

In [ ]:
# Create the src/ folder on Drive (no error if it already exists)
import os
os.makedirs("/content/drive/MyDrive/DSA4050/ksl-recognition/src", exist_ok=True)
print("src/ folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/DSA4050/ksl-recognition/src/ksl_utils.py
"""
ksl_utils.py
============
Shared helper functions for the KSL Recognition term paper (DSA 4050-XA).

Every notebook imports this file so that figures, tables, models and results
are saved in the SAME place with the SAME naming style. That makes writing
the final report easy: everything the report needs is already in outputs/.

Author : Selmah Tzindori (669602)
Course : DSA 4050-XA Deep Learning, USIU-Africa, Fall 2026
"""

import os
import json
import random
import datetime as dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# 1. PROJECT PATHS
# -----------------------------------------------------------------------------
# On Colab the project lives on Google Drive. If we are NOT on Colab
# (e.g. testing on a laptop) we fall back to a local folder so the code
# still runs. You can also override the location with the KSL_ROOT env var.
DRIVE_ROOT = "/content/drive/MyDrive/DSA4050/ksl-recognition"
LOCAL_ROOT = os.path.join(os.path.expanduser("~"), "DSA4050", "ksl-recognition")
PROJECT_ROOT = os.environ.get(
    "KSL_ROOT", DRIVE_ROOT if os.path.isdir("/content/drive") else LOCAL_ROOT
)

# Every folder the project uses, in one dictionary.
PATHS = {
    "root":       PROJECT_ROOT,
    "raw":        os.path.join(PROJECT_ROOT, "data", "raw"),        # original downloads, never edited
    "processed":  os.path.join(PROJECT_ROOT, "data", "processed"),  # cleaned / resized images, splits
    "models":     os.path.join(PROJECT_ROOT, "models"),             # trained models (.keras, .pkl)
    "checkpoints":os.path.join(PROJECT_ROOT, "models", "checkpoints"),  # best weights saved during training
    "figures":    os.path.join(PROJECT_ROOT, "outputs", "figures"), # .png charts for report and slides
    "tables":     os.path.join(PROJECT_ROOT, "outputs", "tables"),  # .csv tables for report
    "logs":       os.path.join(PROJECT_ROOT, "outputs", "logs"),    # training histories, run logs
    "src":        os.path.join(PROJECT_ROOT, "src"),                # this helper file lives here
}

# Raw dataset backup on Drive (one zip) and its fast local copy on Colab's disk.
# Reading thousands of small images straight from Drive is very slow, so each
# session we unzip the Drive backup onto Colab's local disk and work from there.
RAW_ZIP = os.path.join(PATHS["raw"], "kslc_dataset.zip")
LOCAL_DATA = "/content/ksl_data" if os.path.isdir("/content") else os.path.join(PROJECT_ROOT, "_local_data")

# Pre-processed (cropped + resized) images: zip on Drive, unzipped locally each session
PROCESSED_ZIP = os.path.join(PATHS["processed"], "ksl_224.zip")
LOCAL_PROCESSED = "/content/ksl_224" if os.path.isdir("/content") else os.path.join(PROJECT_ROOT, "_local_224")

RESULTS_FILE = os.path.join(PROJECT_ROOT, "outputs", "results.json")         # key numbers for the report
EXPERIMENTS_FILE = os.path.join(PROJECT_ROOT, "outputs", "experiments.csv")  # one row per trained model

SEED = 42  # one seed for the whole project so results can be reproduced


def make_folders(verbose: bool = True) -> None:
    """Create every project folder (safe to run many times; existing folders are kept)."""
    for name, path in PATHS.items():
        os.makedirs(path, exist_ok=True)
        if verbose:
            print(f"  ✔ {name:<12} {path}")


def restore_raw_data(local_dir: str = LOCAL_DATA) -> str:
    """
    Make the raw dataset available on Colab's fast local disk.
    - If it is already unzipped this session, do nothing.
    - Otherwise copy the zip from Drive and unzip it (about 1-2 minutes).
    Returns the local folder that holds the dataset.
    """
    import shutil, time, zipfile
    if os.path.isdir(local_dir) and os.listdir(local_dir):
        print(f"Data already available at {local_dir}")
        return local_dir
    if not os.path.exists(RAW_ZIP):
        raise FileNotFoundError(f"{RAW_ZIP} not found. Run 01_data_collection first.")
    start = time.time()
    tmp_zip = os.path.join(os.path.dirname(local_dir), "kslc_dataset_tmp.zip")
    shutil.copy(RAW_ZIP, tmp_zip)                  # Drive -> local disk (one big file is fast)
    with zipfile.ZipFile(tmp_zip) as z:
        z.extractall(local_dir)                    # unzip on local disk
    os.remove(tmp_zip)                             # free space
    print(f"Data restored to {local_dir} in {time.time() - start:.0f}s")
    return local_dir


def restore_processed_data(local_dir: str = LOCAL_PROCESSED) -> str:
    """
    Make the pre-processed 224x224 images (made in 03_preprocessing) available locally.
    Same idea as restore_raw_data(), but for the much smaller processed zip.
    """
    import shutil, time, zipfile
    if os.path.isdir(local_dir) and os.listdir(local_dir):
        print(f"Processed data already available at {local_dir}")
        return local_dir
    if not os.path.exists(PROCESSED_ZIP):
        raise FileNotFoundError(f"{PROCESSED_ZIP} not found. Run 03_preprocessing first.")
    start = time.time()
    tmp_zip = os.path.join(os.path.dirname(local_dir), "ksl_224_tmp.zip")
    shutil.copy(PROCESSED_ZIP, tmp_zip)
    with zipfile.ZipFile(tmp_zip) as z:
        z.extractall(local_dir)
    os.remove(tmp_zip)
    print(f"Processed data restored to {local_dir} in {time.time() - start:.0f}s")
    return local_dir


def load_split_catalogue():
    """Read the final catalogue (one row per kept image, with its train/val/test split)."""
    return pd.read_csv(os.path.join(PATHS["processed"], "catalogue_split.csv"))


def load_class_names() -> list:
    """Sign names in label-index order (index 0 = first name), saved by 03_preprocessing."""
    with open(os.path.join(PATHS["processed"], "class_names.json")) as f:
        return json.load(f)


# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY
# -----------------------------------------------------------------------------
def set_seed(seed: int = SEED) -> None:
    """Fix all random seeds so every run gives the same splits and (near) same results."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import tensorflow as tf  # only if TensorFlow is installed
        tf.random.set_seed(seed)
    except ImportError:
        pass
    print(f"Random seed set to {seed}")


def _timestamp() -> str:
    """Current date-time as text, used to stamp every saved result."""
    return dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# -----------------------------------------------------------------------------
# 3. SAVING OUTPUTS FOR THE REPORT
# -----------------------------------------------------------------------------
def save_figure(fig, name: str, notebook: str, caption: str = "", dpi: int = 300) -> str:
    """
    Save a matplotlib figure as a high-resolution PNG in outputs/figures/.

    name     : short file name, e.g. "02_class_distribution"
    notebook : which notebook made it, e.g. "02_eda" (recorded in the log)
    caption  : one sentence describing the figure (re-used in the report)
    """
    path = os.path.join(PATHS["figures"], f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    _log_output("figure", name, path, notebook, caption)
    print(f"Figure saved -> {path}")
    return path


def save_table(df: pd.DataFrame, name: str, notebook: str, caption: str = "", index: bool = False) -> str:
    """Save a pandas DataFrame as CSV in outputs/tables/ (opens in Excel/Word for the report)."""
    path = os.path.join(PATHS["tables"], f"{name}.csv")
    df.to_csv(path, index=index)
    _log_output("table", name, path, notebook, caption)
    print(f"Table saved  -> {path}")
    return path


def log_result(key: str, value, notebook: str) -> None:
    """
    Store one key number/fact for the report in outputs/results.json.
    Example: log_result("n_images_total", 6249, "01_data")
    Running it again with the same key overwrites the old value.
    """
    results = load_results()
    results[key] = {"value": value, "notebook": notebook, "saved_at": _timestamp()}
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Result logged: {key} = {value}")


def load_results() -> dict:
    """Read every logged result (empty dict if nothing logged yet)."""
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            return json.load(f)
    return {}


def log_experiment(model_name: str, model_type: str, metrics: dict, notebook: str,
                   params: dict | None = None, notes: str = "") -> pd.DataFrame:
    """
    Add one row to outputs/experiments.csv for every trained model.
    This table becomes the model-comparison table in the report.

    model_type : "classic_ml" or "deep_learning"
    metrics    : e.g. {"accuracy": 0.91, "macro_f1": 0.90, "train_time_s": 120}
    params     : hyper-parameters used, e.g. {"epochs": 20, "lr": 1e-3}
    If the same model_name already exists, its row is replaced.
    """
    row = {"model": model_name, "type": model_type, **metrics,
           "params": json.dumps(params or {}), "notes": notes,
           "notebook": notebook, "saved_at": _timestamp()}
    if os.path.exists(EXPERIMENTS_FILE):
        df = pd.read_csv(EXPERIMENTS_FILE)
        df = df[df["model"] != model_name]             # drop old run of the same model
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(EXPERIMENTS_FILE, index=False)
    print(f"Experiment logged: {model_name} -> {metrics}")
    return df


def _log_output(kind: str, name: str, path: str, notebook: str, caption: str) -> None:
    """Keep a list of every saved figure/table (outputs/logs/outputs_index.csv) with captions."""
    index_file = os.path.join(PATHS["logs"], "outputs_index.csv")
    row = pd.DataFrame([{"kind": kind, "name": name, "path": path, "notebook": notebook,
                         "caption": caption, "saved_at": _timestamp()}])
    if os.path.exists(index_file):
        old = pd.read_csv(index_file)
        old = old[~((old["kind"] == kind) & (old["name"] == name))]
        row = pd.concat([old, row], ignore_index=True)
    row.to_csv(index_file, index=False)


# -----------------------------------------------------------------------------
# 4. SAVING MODELS AND TRAINING HISTORY
# -----------------------------------------------------------------------------
def model_path(name: str, ext: str = "keras") -> str:
    """Full Drive path for a model file, e.g. model_path("cnn_scratch") -> .../models/cnn_scratch.keras"""
    return os.path.join(PATHS["models"], f"{name}.{ext}")


def save_history(history, name: str, notebook: str) -> str:
    """Save a Keras training history (loss/accuracy per epoch) to outputs/logs/ as CSV."""
    hist = history.history if hasattr(history, "history") else history
    df = pd.DataFrame(hist)
    df.index.name = "epoch"
    path = os.path.join(PATHS["logs"], f"history_{name}.csv")
    df.to_csv(path)
    print(f"History saved -> {path}")
    return path


def plot_history(history, name: str, notebook: str):
    """Plot and save training vs validation accuracy and loss curves side by side."""
    hist = history.history if hasattr(history, "history") else history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        if metric in hist:
            ax.plot(hist[metric], label="train")
        if f"val_{metric}" in hist:
            ax.plot(hist[f"val_{metric}"], label="validation")
        ax.set_title(f"{name}: {metric}")
        ax.set_xlabel("epoch")
        ax.set_ylabel(metric)
        ax.legend()
    fig.tight_layout()
    save_figure(fig, f"history_{name}", notebook, caption=f"Training curves for {name}.")
    return fig


# -----------------------------------------------------------------------------
# 5. QUICK SUMMARY
# -----------------------------------------------------------------------------
def project_summary() -> None:
    """Print what has been saved so far: number of figures, tables, models and results."""
    def count(folder, ext):
        return len([f for f in os.listdir(folder) if f.endswith(ext)]) if os.path.isdir(folder) else 0
    print("Project root :", PROJECT_ROOT)
    print("Figures      :", count(PATHS["figures"], ".png"))
    print("Tables       :", count(PATHS["tables"], ".csv"))
    print("Models       :", count(PATHS["models"], ".keras") + count(PATHS["models"], ".pkl"))
    print("Results      :", len(load_results()))
    n_exp = len(pd.read_csv(EXPERIMENTS_FILE)) if os.path.exists(EXPERIMENTS_FILE) else 0
    print("Experiments  :", n_exp)


## Step 3 · Import the helpers and create the folders
We add `src/` to Python's search path so `import ksl_utils` works, then create every project folder.

In [ ]:
# Make ksl_utils importable from Drive
import sys
sys.path.insert(0, "/content/drive/MyDrive/DSA4050/ksl-recognition/src")

# Import the helpers (reload in case the file was just rewritten)
import importlib
import ksl_utils as ku
importlib.reload(ku)

# Create the full folder structure and show it
print("Creating project folders:")
ku.make_folders()

# Fix random seeds for reproducibility
ku.set_seed()

## Step 4 · Check the GPU
Deep learning models train **much** faster on a GPU.
If the output says *No GPU found*, go to **Runtime → Change runtime type → T4 GPU → Save**, then re-run the notebook from the top.

In [ ]:
# Check which GPU (if any) Colab has given us
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {len(gpus)} device(s)")
    !nvidia-smi --query-gpu=name,memory.total --format=csv
else:
    print("No GPU found. Switch the runtime to T4 GPU (Runtime -> Change runtime type).")

## Step 5 · Record the environment
The report's *Methodology* section should state the tools used. We log them now so they are ready later.

In [ ]:
# Collect version and hardware information
import platform, sklearn, numpy as np, pandas as pd

NOTEBOOK = "00_project_setup"
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null
env = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "gpu": gpu_name[0] if gpus and gpu_name else "none",
    "platform": "Google Colab",
}

# Save each item to results.json and show them as a table
for key, value in env.items():
    ku.log_result(f"env_{key}", value, NOTEBOOK)

env_df = pd.DataFrame(env.items(), columns=["component", "version"])
ku.save_table(env_df, "00_environment", NOTEBOOK, caption="Software and hardware environment used for the project.")
env_df

## Step 6 · Standard first cell for every notebook
Every project notebook (`01_`, `02_`, …) starts with this cell. Only `NOTEBOOK` changes.

```python
# =============================================================================
# STANDARD SETUP CELL — the first code cell of EVERY project notebook
# =============================================================================
# 1. Connect Google Drive (where the data, models and outputs live)
from google.colab import drive
drive.mount("/content/drive")

# 2. Make the shared helper file (ksl_utils.py) importable
import sys
sys.path.insert(0, "/content/drive/MyDrive/DSA4050/ksl-recognition/src")

# 3. Import helpers and fix the random seed for reproducibility
import ksl_utils as ku
ku.set_seed()

# 4. Name of THIS notebook — every saved figure/table/result is tagged with it
NOTEBOOK = "XX_notebook_name"   # <-- change in each notebook, e.g. "01_data_collection"

# 5. Shortcuts to the project folders
PATHS = ku.PATHS
print("Project root:", ku.PROJECT_ROOT)
```

A ready-made template with this cell and all section headings is in `notebooks/_TEMPLATE.ipynb`: make a copy of it for each new notebook.

## Step 7 · Documentation rules for this project
These keep the notebooks report-ready:

1. **Header block**: each notebook opens with *Objective*, *Inputs*, *Outputs*.
2. **One markdown cell before each step** explaining *what* we are doing and *why*.
3. **Comments on every code block**: what each line does in plain language.
4. **Save everything the report may need**: `save_figure` for charts, `save_table` for tables, `log_result` for single numbers, `log_experiment` for each model.
5. **Name files with the notebook number first**, e.g. `02_class_distribution.png`, so it's obvious where each output came from.
6. **Interpretation after every output**: a short markdown cell saying what the chart/table shows.
7. **Findings for the report**: each notebook ends with 3–5 bullet points ready to paste into the write-up.
8. **Before committing**: *Runtime → Restart and run all*, so the saved notebook shows clean, in-order outputs. Then download it (*File → Download → .ipynb*) into the repo's `notebooks/` folder.

## Check · Project status

In [ ]:
# Confirm everything is in place
ku.project_summary()

## Summary
- Google Drive is connected and the project folders exist.
- `ksl_utils.py` is saved in `src/` and imports correctly.
- The environment (Python, TensorFlow, GPU) is recorded in `results.json` and `outputs/tables/00_environment.csv`.

**Next notebook:** `01_data_collection.ipynb`: download the KSL dataset from Zindi into `data/raw/`.